# TC Explorer 2.0 — Project Pipeline Demonstration

This notebook demonstrates the **TD-DECA 2.0** data pipeline end-to-end using the project's own
functions — `DataManager`, `Ingestor`, the BARPA/CCAM loaders, `standardize_records`, and
`validate_records` — instead of ad-hoc parsing scripts.

Unlike the exploratory `plot_ccam_cdd_periods_split_tracks` notebook (which reimplemented CSV
parsing, gap-splitting, and unit conversion by hand), every step below calls the actual
application code in `app/`. This is intended as a client-facing walkthrough: if the pipeline
code changes, this notebook's outputs change with it, so it always reflects the real system.

**Pipeline stages shown:**
1. Load raw BARPA and CCAM CSV files via `DataManager`
2. Inspect the standardised `TCRecord` objects it produces
3. Run the project validator and review the data-quality report
4. Build tidy per-track and per-point DataFrames (same shape the dashboard uses)
5. Reproduce the client-relevant analyses: cyclones per period, tracks map, intensity
   distribution, genesis/track density map, and a longevity audit


## 1. Setup

Add the project root to `sys.path` so the notebook can import `app.*` exactly as the dashboard does.

In [3]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "app").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)


Project root: c:\Users\larev\OneDrive\Documents\GitHub\ITProject2


In [4]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from app.services.data_manager import DataManager
from app.pipeline.validators import validate_records, validate_record

pd.set_option("display.max_columns", 50)

DATA_DIR = PROJECT_ROOT / "data"
data_manager = DataManager(data_dir=DATA_DIR)

print("Available source files:")
for name in data_manager.list_datasets():
    print(" -", name)


Available source files:
 - barpa_cdd_all_ssp370.csv
 - barpa_te_all_ssp370.csv
 - ccam_cdd_all_ssp370.csv
 - ccam_te_all_ssp370.csv


## 2. Ingest datasets via the project pipeline

Each call below goes through `DataManager.load_dataset()` → `Ingestor.ingest()` → the correct
loader (`BARPALoader` / `CCAMLoader`) → `standardize_records()` → `validate_records()`.
No parsing logic is duplicated here — this is the same code path the dashboard uses.


In [5]:
DATASET_FILES = {
    "BARPA / CDD": ("barpa_cdd_all_ssp370.csv", "barpa"),
    "BARPA / TE": ("barpa_te_all_ssp370.csv", "barpa"),
    "CCAM / CDD": ("ccam_cdd_all_ssp370.csv", "ccam"),
    "CCAM / TE": ("ccam_te_all_ssp370.csv", "ccam"),
}

ingest_results = {}

for label, (filename, dataset_type) in DATASET_FILES.items():
    filepath = DATA_DIR / filename
    if not filepath.exists():
        print(f"Skipping {label}: {filename} not found in {DATA_DIR}")
        continue

    result = data_manager.load_dataset(filename, dataset_type=dataset_type, force_reload=True)
    ingest_results[label] = result
    print(f"{label:14s} -> {len(result['records']):5d} usable records "
          f"(loader: {result['loader']})")


BARPA / CDD    -> 15628 usable records (loader: BARPALoader)
BARPA / TE     ->   203 usable records (loader: BARPALoader)
CCAM / CDD     -> 16741 usable records (loader: CCAMLoader)
CCAM / TE      ->   129 usable records (loader: CCAMLoader)


## 3. Data-quality report from the project validator

`validate_records()` is the same function used by `Ingestor.ingest()`. It rejects tracks with
impossible lifetimes (e.g. a raw `TrackID` reused across unrelated storms), missing coordinates,
or out-of-range categories. The table below summarises the report for every dataset that was
loaded above.


In [6]:
quality_rows = []

for label, result in ingest_results.items():
    validation = result["validation"]
    quality_rows.append({
        "dataset": label,
        "total_records": validation.get("total_records"),
        "usable_records": validation.get("usable_records"),
        "invalid_count": validation.get("invalid_count"),
        "total_points": validation.get("total_points"),
    })

quality_df = pd.DataFrame(quality_rows)
quality_df


,dataset,total_records,usable_records,invalid_count,total_points
0,BARPA / CDD,15869,15628,241,404635
1,BARPA / TE,16134,203,15931,411622
2,CCAM / CDD,16836,16734,102,374308
3,CCAM / TE,15375,113,15262,382190


In [7]:
# Drill into the specific issues found, aggregated across all loaded datasets
from collections import Counter

issue_totals = Counter()
for result in ingest_results.values():
    issue_totals.update(result["validation"].get("issue_counts", {}))

issues_df = (
    pd.Series(issue_totals, name="occurrences")
    .sort_values(ascending=False)
    .rename_axis("issue")
    .reset_index()
)
issues_df


,issue,occurrences
0,Track has fewer than 2 valid points.,756
1,Mean translation speed is 611.8 km/h; check tr...,17
2,Mean translation speed is 515.8 km/h; check tr...,15
3,Mean translation speed is 522.6 km/h; check tr...,14
4,Mean translation speed is 636.0 km/h; check tr...,14
...,...,...
9822,Mean translation speed is 1201.1 km/h; check t...,1
9823,Mean translation speed is 1084.4 km/h; check t...,1
9824,Mean translation speed is 1943.4 km/h; check t...,1
9825,Mean translation speed is 1087.7 km/h; check t...,1


## 4. Build tidy DataFrames from `TCRecord` objects

This mirrors exactly what `dashboard.py` does when it flattens `TCRecord` → `TRACKS` / `POINTS`.
Reusing this logic here means the notebook and the live dashboard can never silently disagree.


In [8]:
def records_to_frames(records):
    track_rows = []
    point_rows = []

    for record in records:
        track_rows.append({
            "track_id": record.track_id,
            "raw_track_id": record.metadata.get("raw_track_id"),
            "dataset": record.model or record.dataset_id,
            "tracker": record.tracker or "unknown",
            "season": record.metadata.get("season"),
            "scenario": record.scenario or "unknown",
            "region": record.region or "unknown",
            "year": record.year,
            "genesis_time": record.genesis_time,
            "genesis_lat": record.genesis_lat,
            "genesis_lon": record.genesis_lon,
            "landfall": bool(record.landfall),
            "lifetime_hours": record.lifetime_hours or 0.0,
            "max_wind_speed": record.max_wind_speed or 0.0,
            "min_pressure": record.min_pressure,
            "max_category": record.max_category or 0,
            "point_count": len(record.points),
        })

        for step, point in enumerate(record.points):
            point_rows.append({
                "track_id": record.track_id,
                "step": step,
                "time": point.time,
                "lat": point.lat,
                "lon": point.lon,
                "wind_speed": point.wind_speed if point.wind_speed is not None else 0.0,
                "category": point.category if point.category is not None else 0,
            })

    tracks_df = pd.DataFrame(track_rows)
    points_df = pd.DataFrame(point_rows)
    return tracks_df, points_df


all_tracks = []
all_points = []

for label, result in ingest_results.items():
    tracks_df, points_df = records_to_frames(result["records"])
    tracks_df["dataset_label"] = label
    points_df["dataset_label"] = label
    all_tracks.append(tracks_df)
    all_points.append(points_df)

TRACKS = pd.concat(all_tracks, ignore_index=True) if all_tracks else pd.DataFrame()
POINTS = pd.concat(all_points, ignore_index=True) if all_points else pd.DataFrame()

print("Total tracks:", len(TRACKS))
print("Total points:", len(POINTS))
TRACKS.head()


Total tracks: 32701
Total points: 781022


,track_id,raw_track_id,dataset,tracker,season,scenario,region,year,genesis_time,genesis_lat,genesis_lon,landfall,lifetime_hours,max_wind_speed,min_pressure,max_category,point_count,dataset_label
0,barpa_cdd_all_ssp370|ERA5|CDD|season-1981|trac...,32,BARPA,CDD,1981,historical,Australia,1980,1980-12-17 18:00:00+00:00,-14.0,121.1,False,186.0,133.20,952.7,1,30,BARPA / CDD
1,barpa_cdd_all_ssp370|ERA5|CDD|season-1981|trac...,33,BARPA,CDD,1981,historical,Australia,1981,1981-01-12 12:00:00+00:00,-12.0,100.1,False,84.0,72.72,992.2,0,15,BARPA / CDD
2,barpa_cdd_all_ssp370|ERA5|CDD|season-1981|trac...,34,BARPA,CDD,1981,historical,Australia,1981,1981-01-17 00:00:00+00:00,-16.7,155.6,False,114.0,99.00,981.3,0,20,BARPA / CDD
3,barpa_cdd_all_ssp370|ERA5|CDD|season-1981|trac...,35,BARPA,CDD,1981,historical,Australia,1981,1981-01-25 06:00:00+00:00,-29.9,170.2,False,150.0,83.16,988.2,0,24,BARPA / CDD
4,barpa_cdd_all_ssp370|ERA5|CDD|season-1981|trac...,36,BARPA,CDD,1981,historical,Australia,1981,1981-02-02 06:00:00+00:00,-14.2,118.6,False,120.0,123.48,966.6,1,21,BARPA / CDD


## 5. Cyclones per year, by dataset

Frequency comparison across BARPA/CCAM and CDD/TE, using the genesis year derived by
`TCRecord.derive_fields()` in the pipeline (not recomputed here).


In [9]:
frequency = (
    TRACKS.groupby(["year", "dataset_label"], as_index=False)
    .size()
    .rename(columns={"size": "cyclones"})
)

fig_frequency = px.line(
    frequency,
    x="year",
    y="cyclones",
    color="dataset_label",
    markers=True,
    title="Cyclones per year by dataset and tracker",
)
fig_frequency.update_layout(template="plotly_white", height=420)
fig_frequency.show()


ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

## 6. Intensity distribution

Maximum category reached, computed from `TCRecord.max_category` (derived by the pipeline from
wind speed already converted to km/h during loading — not recalculated here).


In [ ]:
intensity = (
    TRACKS.groupby(["max_category", "dataset_label"], as_index=False)
    .size()
    .rename(columns={"size": "cyclones"})
)

fig_intensity = px.bar(
    intensity,
    x="max_category",
    y="cyclones",
    color="dataset_label",
    barmode="group",
    title="Maximum category distribution by dataset and tracker",
)
fig_intensity.update_layout(template="plotly_white", height=420)
fig_intensity.show()


## 7. Track map for a sample of strongest cyclones

Selects the highest-category tracks per dataset and plots their paths, using the
already-standardised `lat`/`lon`/`category` fields from `POINTS`.


In [ ]:
CATEGORY_COLOURS = {
    0: "#94a3b8", 1: "#38bdf8", 2: "#22c55e",
    3: "#facc15", 4: "#fb923c", 5: "#ef4444",
}

top_tracks = (
    TRACKS.sort_values(["max_category", "max_wind_speed"], ascending=False)
    .groupby("dataset_label")
    .head(3)["track_id"]
)

fig_map = go.Figure()

for track_id in top_tracks:
    track_points = POINTS[POINTS["track_id"] == track_id].sort_values("step")
    if track_points.empty:
        continue

    category = int(TRACKS.loc[TRACKS["track_id"] == track_id, "max_category"].iloc[0])
    fig_map.add_trace(go.Scattergeo(
        lon=track_points["lon"],
        lat=track_points["lat"],
        mode="lines+markers",
        line=dict(width=2, color=CATEGORY_COLOURS.get(category, "#94a3b8")),
        marker=dict(size=4),
        name=str(track_id),
    ))

fig_map.update_geos(
    projection_type="mercator",
    lonaxis_range=[100, 180],
    lataxis_range=[-45, 0],
    showland=True, landcolor="#e9eef3",
    showocean=True, oceancolor="#eaf7fb",
    showcoastlines=True, coastlinecolor="#94a3b8",
)
fig_map.update_layout(
    title="Sample of strongest cyclone tracks per dataset",
    height=520, template="plotly_white",
)
fig_map.show()


## 8. Genesis and track density map

A 2D histogram of all standardised track points, showing where cyclones spend the most time —
useful for regional hazard assessment comparisons across datasets.


In [ ]:
fig_density = px.density_heatmap(
    POINTS,
    x="lon",
    y="lat",
    facet_col="dataset_label",
    nbinsx=40,
    nbinsy=30,
    title="Track point density by dataset and tracker",
    color_continuous_scale="YlOrRd",
)
fig_density.update_layout(height=420, template="plotly_white")
fig_density.show()


## 9. Longevity audit

The validator already rejects impossible lifetimes before records reach this notebook, so this
chart doubles as a sanity check that the pipeline is working: no bar here should exceed
60 days (1,440 hours).


In [ ]:
MAX_PLAUSIBLE_HOURS = 24 * 60

longevity_summary = (
    TRACKS.groupby("dataset_label")["lifetime_hours"]
    .agg(["count", "mean", "median", "max"])
    .rename(columns={"count": "tracks", "mean": "mean_hours", "median": "median_hours", "max": "max_hours"})
    .reset_index()
)

longevity_summary["exceeds_plausible_limit"] = longevity_summary["max_hours"] > MAX_PLAUSIBLE_HOURS
longevity_summary


In [ ]:
fig_longevity = px.histogram(
    TRACKS,
    x="lifetime_hours",
    color="dataset_label",
    nbins=40,
    title="Track lifetime distribution (hours) — validator caps this at 1,440 h",
)
fig_longevity.add_vline(x=MAX_PLAUSIBLE_HOURS, line_dash="dash", line_color="red")
fig_longevity.update_layout(template="plotly_white", height=420)
fig_longevity.show()


## 10. Export for client review

Writes the same two CSV shapes the dashboard's "Download CSV" button produces — one cyclone
summary file, one track-points file — generated directly from the pipeline's `TCRecord` output.


In [ ]:
OUTPUT_DIR = PROJECT_ROOT / "output"
OUTPUT_DIR.mkdir(exist_ok=True)

summary_path = OUTPUT_DIR / "tc_pipeline_cyclone_summary.csv"
points_path = OUTPUT_DIR / "tc_pipeline_track_points.csv"

TRACKS.to_csv(summary_path, index=False)
POINTS.to_csv(points_path, index=False)

print("Saved:", summary_path)
print("Saved:", points_path)
